# 5.2 · Softmax 回归 / Softmax (Multinomial Logistic) Regression

> **课程定位 / Where this fits**
> 5.1 的逻辑回归只能二分类。现实里 ≥3 类(鸢尾花 3 种、手写数字 10 种)很常见。Softmax 回归把 sigmoid **推广到 K 类**, 输出一个**概率分布**。它也是神经网络最后一层的标配(Part 12 会再见)。
> Softmax generalises logistic regression to K classes — and it's the standard output layer of neural nets.

> 💡 **面试相关 / Interview-relevant**
> - "softmax 公式 + 为什么要减最大值(数值稳定)" ★★★★
> - "softmax + 交叉熵的梯度为什么是 (p - y)" ★★★★
> - "softmax(K类) vs K 个 one-vs-rest 逻辑回归 区别" ★★★★
> - "softmax 为什么过参数化 / 为什么常固定一类" ★★★

---

## 学习目标 / Learning Objectives
1. 从二分类 sigmoid 推广到 K 类 **softmax**。
2. **数值稳定**实现(减最大值)。
3. 交叉熵损失 + 梯度 = $(\mathbf{p}-\mathbf{y})$ 的优雅结构(承接 5.1)。
4. 从零实现 vs sklearn `multinomial`。
5. softmax vs OvR 的区别(为 5.13 铺垫)。

## 目录 / TOC
1. [从 sigmoid 到 softmax ⭐](#1)
2. [🌸 数据: Iris](#2)
3. [数值稳定的 softmax ⭐](#3)
4. [从零实现](#4)
5. [对照 sklearn + 决策边界](#5)
6. [softmax vs OvR](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 从 sigmoid 到 softmax ⭐ / From Sigmoid to Softmax

二分类时 sigmoid 给出 $\Pr(y=1)$, 另一类是 $1-p$。K 类时, 每类一组权重 $\mathbf{w}_k$, 先算 K 个**logit(分数)** $z_k=\mathbf{x}^\top\mathbf{w}_k$, 再用 **softmax** 归一化成概率分布：

$$\Pr(y=k\mid\mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

性质: 全部 $>0$、加起来 $=1$、**单调放大差距**(指数让大的更大 → "soft" 版的 argmax)。

**损失**: 多类交叉熵(= 多项分布 MLE, 承接 2.9/5.1)。设真实类 one-hot 为 $\mathbf{y}$：
$$J = -\frac{1}{n}\sum_i \sum_k y_{ik}\log p_{ik}$$

**梯度**(和 5.1 一模一样的优雅形式)：
$$\nabla_{\mathbf{w}_k} J = \frac{1}{n}\sum_i (p_{ik}-y_{ik})\,\mathbf{x}_i = \frac{1}{n}\mathbf{X}^\top(\mathbf{P}-\mathbf{Y})_{:,k}$$

**过参数化**: softmax 有冗余——给所有 $\mathbf{w}_k$ 同时加常向量, 概率不变。所以可固定一类权重为 0(二分类的 sigmoid 正是 K=2 的特例)。


<a id="2"></a>
## 2. 数据: Iris 鸢尾花 / The Iris Dataset

**经典中的经典**(Fisher 1936)。150 朵鸢尾花, 3 个品种各 50 朵, 4 个特征(花萼/花瓣的长宽)。三分类的标准教学集。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)

iris = load_iris()
X, y = iris.data, iris.target
df = pd.DataFrame(X, columns=iris.feature_names); df["species"] = [iris.target_names[i] for i in y]
print("Iris:", X.shape, "| 3 类各:", np.bincount(y))
print(df.groupby("species").mean().round(2))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)


<a id="3"></a>
## 3. 数值稳定的 softmax ⭐ / Numerically Stable Softmax

朴素 $e^{z_k}$ 在 $z$ 较大时**溢出**(`inf`)。技巧: 分子分母同乘 $e^{-\max_j z_j}$, 结果不变但指数参数 $\le 0$ → 不溢出。**面试常考**。


In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)   # 减每行最大值 → 数值稳定, 结果不变
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

Z_demo = np.array([[1000., 1001., 1002.]])   # 朴素 exp 会 inf
print("稳定 softmax:", softmax(Z_demo), " 和 =", softmax(Z_demo).sum())
print("朴素 exp(1002) =", np.exp(1002.), "→ inf, 故必须减最大值")


<a id="4"></a>
## 4. 从零实现 / From Scratch


In [ ]:
def one_hot(y, K):
    Y = np.zeros((len(y), K)); Y[np.arange(len(y)), y] = 1; return Y

def fit_softmax(X, y, K, lr=0.5, n_iter=2000):
    Xb = np.c_[np.ones(len(X)), X]
    Y = one_hot(y, K)
    W = np.zeros((Xb.shape[1], K))
    for _ in range(n_iter):
        P = softmax(Xb @ W)
        grad = Xb.T @ (P - Y) / len(y)     # (p - y) 结构, 同 5.1
        W -= lr * grad
    return W

W = fit_softmax(Xtr, y_tr, K=3)
Pte = softmax(np.c_[np.ones(len(Xte)), Xte] @ W)
acc = (Pte.argmax(1) == y_te).mean()
print(f"从零 softmax test 准确率: {acc:.3f}")
print("预测概率示例(每行和=1):", Pte[0].round(3), "→ 预测类", Pte[0].argmax())


<a id="5"></a>
## 5. 对照 sklearn + 决策边界 / sklearn & Boundaries

sklearn 的 `LogisticRegression` 默认就用 multinomial(真 softmax) 处理多类。


In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)
print(f"sklearn softmax test 准确率: {clf.score(Xte, y_te):.3f}")
print(f"从零模型与 sklearn 预测一致率: {(Pte.argmax(1) == clf.predict(Xte)).mean():.3f}")

# 2D 决策边界 (用花瓣长宽) / boundaries on 2 features
X2tr = Xtr[:, 2:4]
clf2 = LogisticRegression(max_iter=2000).fit(X2tr, y_tr)
xx, yy = np.meshgrid(np.linspace(X2tr[:,0].min()-.5, X2tr[:,0].max()+.5, 300),
                     np.linspace(X2tr[:,1].min()-.5, X2tr[:,1].max()+.5, 300))
Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
sc_ = ax.scatter(X2tr[:,0], X2tr[:,1], c=y_tr, cmap="viridis", edgecolor="k", s=30)
ax.set_xlabel("petal length (std)"); ax.set_ylabel("petal width (std)")
ax.set_title("Softmax 多类决策边界 — 3 类两两间是直线\n(每类一个线性判别, 分界是分段线性)")
plt.tight_layout(); plt.show()


<a id="6"></a>
## 6. softmax vs OvR / Softmax vs One-vs-Rest

两种做多分类的思路(5.13 会展开)：

| | Softmax (multinomial) | One-vs-Rest (OvR) |
|---|---|---|
| 模型 | **一个**模型, K 组权重联合训练 | **K 个**独立二分类器 |
| 概率 | 天然归一化(和=1) | 各自 sigmoid, 需手动归一化 |
| 适用 | 互斥单标签 | 可扩展到多标签 |
| 直觉 | 类别**竞争**同一份概率 | 每类**独立**判断"是不是我" |

注: sklearn 的 OvR 会把各 sigmoid 概率事后归一化使和=1; 但它仍是 K 个独立边界, 协调性不如 softmax 联合训练(下方实测 OvR 准确率略低)。

互斥单标签问题(如 Iris)优先 softmax; 多标签(一个样本可属多类)用 OvR。


In [ ]:
from sklearn.multiclass import OneVsRestClassifier
ovr = OneVsRestClassifier(LogisticRegression(max_iter=2000)).fit(Xtr, y_tr)
print(f"Softmax 准确率: {clf.score(Xte, y_te):.3f}")
print(f"OvR     准确率: {ovr.score(Xte, y_te):.3f}  (这里略低: 3 个独立边界不如联合训练协调)")
print("softmax 概率天然和=1; OvR 各自 sigmoid 后, sklearn 再做归一化才使和=1:")
print("OvR 概率示例:", ovr.predict_proba(Xte[:1]).round(3), "和=", ovr.predict_proba(Xte[:1]).sum().round(3))


<a id="7"></a>
## 7. 小结 / Summary

```
softmax: p_k = e^{z_k}/Σ e^{z_j} → 概率分布(>0, 和=1), soft 版 argmax
数值稳定: 减每行最大值 (面试常考)
损失=多类交叉熵(=多项 MLE); 梯度=Xᵀ(P-Y), 与 5.1 同结构
过参数化: 可固定一类权重=0; 二分类 sigmoid 是 K=2 特例
softmax(互斥单标签) vs OvR(独立, 可多标签) → 5.13 展开
```

### 💡 面试速查
1. softmax = sigmoid 的 K 类推广, 输出归一化概率分布
2. **减最大值**防溢出, 结果不变
3. 梯度还是 **(p − y)** 的干净形式
4. 互斥多类用 softmax; 多标签用 OvR

### 下一节
**5.3 KNN**——前面都是参数模型(学权重)。KNN 是**非参数惰性**模型: 不训练, 预测时找最近邻投票。
